In [ ]:
import plotly.express as px


import polars as pl

# Fuck pycharm

Apparently pycharm has lost this entire notebook. I guess I'll just tell you what this was.

This notebook was used to recalculate the val and test eval of the nowcasting_temp 1.0 model, which is trained with the MEAN target.
The goal was to see its performance relative to the new FIRST target. To do this, I temporarily patched the _evaluate_model method in aare_train/evaluation/evaluation.py
to refetch the true data from influxdb directly by creating a FeatureSet with a patched WaterTemp instance. The WaterTemp and AirTemp class
were updated to use "mean" agg_fn again, so it's in line with the 1.0 model. I also downgraded darts again to the version the 1.0 model was trained with.

I then called

```
uv run scripts/eval.py nowcasting_temp-1.0 --stride 1 --suffix first-target
```

which created the corrected files (for val, for test you need to adapt again).
Before that I also quickly checked that I can reproduce the eval without any change and I believe I didn't find anything alarming.

I did some sanity checks and then carefully moved all the metrics, raw metrics and forecast_samples around and renamed them.


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("dulwich").setLevel(logging.WARNING)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
original = pl.read_parquet("data/metrics/raw/nowcasting_temp-1.0.parquet")
recreated = pl.read_parquet("data/metrics/raw/nowcasting_temp-1.0-20260516T195016.parquet")

In [ ]:
original.describe()

In [ ]:
recreated.describe()

In [ ]:
cols = ["pred", "actual", "err", "dpd"]
suffix = "_recreated"
df = original.join(recreated, on=["run_ts", "time"], suffix=suffix).select(
    [(pl.col(c) - pl.col(c + suffix)).abs().alias(c + "_diff") for c in cols]
)
px.histogram(df, facet_row=[c + "_diff" for c in cols])